# Save scene as a portable `.nvd` document

The Jupyter-native equivalent of the `nv-ext-save-html` browser demo. `nv.save_document(filename)` serializes the current scene (volumes, meshes, drawings, view settings) to an `.nvd` file (CBOR-encoded NVD format) and triggers a browser download. The file opens in any NiiVue viewer, including [niivue.github.io](https://niivue.github.io/niivue/).

For programmatic use, `await nv.serialize_document()` returns the bytes directly — useful for writing to a controlled path or sending to cloud storage from Python.

Why no `.html` export? `nv-ext-save-html` produces a self-contained HTML file with the niivue library inlined (~1.2 MB). Bundling that into the Python wheel would roughly double its install size for a feature that doesn't fit notebook workflows — the `.nvd` artifact is smaller, opens in any niivue, and Jupyter already provides the share-with-someone affordance. See `CLAUDE.md` for the design rationale.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"

nv = NiiVue(
    background_color=[0.2, 0.2, 0.3, 1.0],
    is_colorbar_visible=True,
    slice_type=3,
    backend="webgl2",
)

filename = widgets.Text(value="scene.nvd", description="Filename")
save_btn = widgets.Button(description="Export .nvd", icon="download", button_style="primary")
status = widgets.Label(value="Loading volume\u2026")

def on_save(_btn):
    name = filename.value.strip() or "scene.nvd"
    nv.save_document(name)
    status.value = f"Triggered browser download: {name}"

save_btn.on_click(on_save)

display(widgets.HBox([filename, save_btn]))
display(nv)
display(status)

nv.add_volume_from_url(
    f"{VOLUMES}/mni152.nii.gz",
    cal_min=30,
    cal_max=80,
    colormap="gray",
)
status.value = "Ready. Adjust the view, then Export to download an .nvd snapshot."


## Programmatic export (no browser download)

`await nv.serialize_document()` returns the raw `.nvd` bytes so Python can write them to a controlled location — useful for batch jobs, automated pipelines, or uploading to object storage.


In [ ]:
from pathlib import Path

data = await nv.serialize_document()
out = Path("/tmp/ipyniivue-scene.nvd")
out.write_bytes(bytes(data))
print(f"Wrote {out} ({out.stat().st_size:,} bytes)")
